# Fine-Grained Influence v4: Clean Sentence-Boundary Alignment

Token-by-token context reveal, but context expansion starts from the **beginning of the preceding sentence** (not the end). This ensures the first tokens of context are sentence-initial, making the intact-vs-shuffled subtraction clean at short distances.

Also includes shuffled control (token-shuffled context, same target).

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Imports OK')

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_finegrain")
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_finegrain")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True
N_DOCS = 150
MAX_CONTEXT = 100
RANDOM_SEED = 42
DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']

print(f"N docs per population: {N_DOCS}")
print(f"Max context: {MAX_CONTEXT} tokens")

In [ ]:
corpus_all = []
with open(DATA_DIR / "raid_corpus.jsonl") as f:
    for line in f:
        corpus_all.append(json.loads(line))

rng = np.random.RandomState(RANDOM_SEED)
sample = []
for pop in ['human', 'ai']:
    pop_docs = [d for d in corpus_all if d['population'] == pop and len(d['text'].split()) >= 250]
    for domain in DOMAINS:
        pool = [d for d in pop_docs if d['domain'] == domain]
        n = min(len(pool), N_DOCS // len(DOMAINS) + 1)
        if n > 0:
            sample.extend(rng.choice(pool, size=n, replace=False))

print(f"Selected {len(sample)} documents")
print(f"  Human: {sum(1 for d in sample if d['population'] == 'human')}")
print(f"  AI: {sum(1 for d in sample if d['population'] == 'ai')}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

In [ ]:
@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def find_sentence_boundaries(token_ids, tokenizer):
    text = tokenizer.decode(token_ids)
    boundaries = [0]
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    pos = 0
    for sent in sentences:
        sent_ids = tokenizer.encode(sent.strip(), add_special_tokens=False)
        pos += len(sent_ids)
        if pos < len(token_ids):
            boundaries.append(pos)
    return boundaries


def compute_token_by_token_curve(doc, shuffled=False, rng_shuf=None):
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    sent_bounds = find_sentence_boundaries(full_ids, tokenizer)

    # Target = sentence at ~65%
    tgt_idx = min(int(len(sent_bounds) * 0.65), len(sent_bounds) - 2)
    if tgt_idx < 2:
        return None
    target_start = sent_bounds[tgt_idx]
    target_end = sent_bounds[tgt_idx + 1] if tgt_idx + 1 < len(sent_bounds) else n
    if target_end - target_start < 3:
        return None

    target_ids = full_ids[target_start:target_end]

    # Context = everything before target_start
    # We expand FORWARD from the start of the preceding sentence
    # Find the start of the sentence just before the target
    prev_sent_start = sent_bounds[tgt_idx - 1] if tgt_idx >= 1 else 0

    # All context tokens, ordered from the preceding sentence boundary forward
    # i.e. context_pool[0] is the first token of the sentence before target,
    # context_pool[1] is the second, etc., going backwards through earlier sentences
    context_pool = list(full_ids[0:target_start])

    if shuffled and rng_shuf is not None:
        # Shuffle all context tokens
        context_pool_shuffled = list(context_pool)
        rng_shuf.shuffle(context_pool_shuffled)
        context_pool = context_pool_shuffled

    max_ctx = min(MAX_CONTEXT, len(context_pool))
    ppls = []
    ctx_lengths = []

    for ctx_len in range(1, max_ctx + 1):
        # Take the LAST ctx_len tokens of the context pool
        # For intact: these are the ctx_len tokens immediately before the target
        # But we want to expand from the preceding sentence boundary FORWARD
        # So we take from prev_sent_start, expanding backward AND forward

        # Actually simpler: take tokens starting from (target_start - ctx_len)
        # but ensure we start at prev_sent_start first, then expand backward
        
        # Rethink: we want the first token added to be the START of the preceding sentence
        # So context grows like:
        #   ctx=1: [prev_sent_start]
        #   ctx=2: [prev_sent_start, prev_sent_start+1]
        #   ...until we reach target_start, then keep going backward

        if ctx_len <= (target_start - prev_sent_start):
            # Still within the preceding sentence, expanding forward from its start
            ctx_start = prev_sent_start
            ctx_tokens = context_pool[ctx_start:ctx_start + ctx_len]
        else:
            # We've covered the full preceding sentence, now expand backward
            remaining = ctx_len - (target_start - prev_sent_start)
            backward_start = max(0, prev_sent_start - remaining)
            ctx_tokens = context_pool[backward_start:target_start]
            # Should be ctx_len tokens
            ctx_tokens = ctx_tokens[-ctx_len:]  # safety trim

        chunk = ctx_tokens + target_ids
        ppl = compute_ppl(chunk, len(ctx_tokens), len(chunk))
        if not math.isinf(ppl):
            ppls.append(ppl)
            ctx_lengths.append(ctx_len)

    if len(ppls) < 10:
        return None

    return {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'population': doc['population'],
        'ctx_lengths': ctx_lengths,
        'ppls': ppls,
        'prev_sent_len': target_start - prev_sent_start,
    }


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "finegrain_results_v4.json"
shuffled_path = BASE_DIR / "finegrain_shuffled_v4.json"

if results_path.exists() and shuffled_path.exists():
    with open(results_path) as f:
        all_curves = json.load(f)
    with open(shuffled_path) as f:
        all_shuffled = json.load(f)
    print(f"Loaded {len(all_curves)} intact + {len(all_shuffled)} shuffled curves")
else:
    all_curves = []
    all_shuffled = []
    rng_shuf = np.random.RandomState(RANDOM_SEED + 99)

    for doc in tqdm(sample, desc="Token-by-token curves"):
        # Intact
        result = compute_token_by_token_curve(doc, shuffled=False)
        if result is None:
            continue
        all_curves.append(result)

        # Shuffled
        result_s = compute_token_by_token_curve(doc, shuffled=True, rng_shuf=rng_shuf)
        if result_s is not None:
            all_shuffled.append(result_s)

    with open(results_path, 'w') as f:
        json.dump(all_curves, f)
    with open(shuffled_path, 'w') as f:
        json.dump(all_shuffled, f)
    print(f"Computed {len(all_curves)} intact + {len(all_shuffled)} shuffled curves")

print(f"Intact — Human: {sum(1 for c in all_curves if c['population'] == 'human')}, AI: {sum(1 for c in all_curves if c['population'] == 'ai')}")
print(f"Shuffled — Human: {sum(1 for c in all_shuffled if c['population'] == 'human')}, AI: {sum(1 for c in all_shuffled if c['population'] == 'ai')}")

# Report preceding sentence lengths
prev_lens = [c['prev_sent_len'] for c in all_curves]
print(f"\nPreceding sentence length: mean={np.mean(prev_lens):.1f}, median={np.median(prev_lens):.0f} tokens")
print("(First phase of context reveal covers this many tokens within the preceding sentence)")

In [ ]:
from scipy.ndimage import uniform_filter1d

common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_mean_curve(curves):
    all_norm = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        if ppl[0] - ppl[-1] > 0:
            norm = (ppl[0] - ppl) / (ppl[0] - ppl[-1])
            interp = np.interp(common_x, ctx, norm, left=np.nan, right=np.nan)
            all_norm.append(interp)
    all_norm = np.array(all_norm)
    mean = np.nanmean(all_norm, axis=0)
    sem = np.nanstd(all_norm, axis=0) / np.sqrt(np.sum(~np.isnan(all_norm), axis=0))
    return mean, sem

def compute_raw_ppl_curve(curves):
    """Mean raw perplexity at each context length (not normalized)."""
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    all_ppl = np.array(all_ppl)
    return np.nanmean(all_ppl, axis=0)

def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p, bc, bm, intercept
    return None

# Split by population
human_intact = [c for c in all_curves if c['population'] == 'human']
ai_intact = [c for c in all_curves if c['population'] == 'ai']
human_shuffled = [c for c in all_shuffled if c['population'] == 'human']
ai_shuffled = [c for c in all_shuffled if c['population'] == 'ai']

# Compute raw ppl curves (not normalized) for subtraction
h_intact_ppl = compute_raw_ppl_curve(human_intact)
h_shuf_ppl = compute_raw_ppl_curve(human_shuffled)
a_intact_ppl = compute_raw_ppl_curve(ai_intact)
a_shuf_ppl = compute_raw_ppl_curve(ai_shuffled)

# Marginals of raw perplexity (negative because ppl decreases with context)
h_intact_marg = -np.diff(h_intact_ppl)  # positive = ppl drops = benefit
h_shuf_marg = -np.diff(h_shuf_ppl)
a_intact_marg = -np.diff(a_intact_ppl)
a_shuf_marg = -np.diff(a_shuf_ppl)

# Corrected marginals
h_corrected_marg = h_intact_marg - h_shuf_marg
a_corrected_marg = a_intact_marg - a_shuf_marg

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# A: Raw perplexity curves — intact vs shuffled
ax = axes[0, 0]
ax.plot(common_x, h_intact_ppl, 'b-', linewidth=2, label='Human intact')
ax.plot(common_x, h_shuf_ppl, 'b:', linewidth=2, label='Human shuffled')
ax.plot(common_x, a_intact_ppl, 'r-', linewidth=2, label='AI intact')
ax.plot(common_x, a_shuf_ppl, 'r:', linewidth=2, label='AI shuffled')
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Perplexity')
ax.set_title('A. Raw Perplexity: Intact vs Shuffled', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# B: Marginal gain — intact vs shuffled
ax = axes[0, 1]
ax.plot(common_x[1:], uniform_filter1d(h_intact_marg, 5), 'b-', linewidth=2, label='Human intact')
ax.plot(common_x[1:], uniform_filter1d(h_shuf_marg, 5), 'b:', linewidth=2, label='Human shuffled')
ax.plot(common_x[1:], uniform_filter1d(a_intact_marg, 5), 'r-', linewidth=2, label='AI intact')
ax.plot(common_x[1:], uniform_filter1d(a_shuf_marg, 5), 'r:', linewidth=2, label='AI shuffled')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Marginal PPL Drop per Token')
ax.set_title('B. Marginal Gain: Intact vs Shuffled', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# C: Corrected marginal (intact - shuffled)
ax = axes[0, 2]
ax.plot(common_x[1:], uniform_filter1d(h_corrected_marg, 5), 'b-', linewidth=2, label='Human (corrected)')
ax.plot(common_x[1:], uniform_filter1d(a_corrected_marg, 5), 'r--', linewidth=2, label='AI (corrected)')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Corrected Marginal (intact - shuffled)')
ax.set_title('C. Pure Coherence Signal', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# D: Power law fit on corrected marginals
ax = axes[1, 0]
for marg, color, label in [(h_corrected_marg, 'blue', 'Human'), (a_corrected_marg, 'red', 'AI')]:
    result = fit_power_law(marg)
    if result:
        slope, r, p, bc, bm, intercept = result
        ax.plot(bc, bm, 'o-', color=color, linewidth=2, markersize=6, label=label)
        fit_x = np.linspace(min(bc), max(bc), 100)
        fit_y = np.exp(intercept) * fit_x ** slope
        ax.plot(fit_x, fit_y, '--', color=color, alpha=0.5,
                label=f'{label}: d^{slope:.2f} (r={r:.2f})')
        print(f"CORRECTED {label}: exponent={slope:.3f}, r={r:.3f}, p={p:.4f}")
ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens, log)')
ax.set_ylabel('Corrected Marginal Benefit')
ax.set_title('D. Power Law Fit (Corrected)', fontweight='bold')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# E: Compare uncorrected vs corrected exponents
ax = axes[1, 1]
# Uncorrected (from intact only, normalized curves)
h_norm_mean, _ = compute_mean_curve(human_intact)
a_norm_mean, _ = compute_mean_curve(ai_intact)
h_uncorr_marg = np.diff(h_norm_mean)
a_uncorr_marg = np.diff(a_norm_mean)

labels_vals = []
for marg, label_prefix, color in [
    (h_uncorr_marg, 'Human uncorrected', 'blue'),
    (h_corrected_marg, 'Human corrected', 'darkblue'),
    (a_uncorr_marg, 'AI uncorrected', 'red'),
    (a_corrected_marg, 'AI corrected', 'darkred'),
]:
    result = fit_power_law(marg)
    if result:
        slope, r, p, bc, bm, intercept = result
        labels_vals.append((label_prefix, slope, r))

x_pos = range(len(labels_vals))
colors = ['#3498db', '#1a5276', '#e74c3c', '#922b21']
ax.bar(x_pos, [v[1] for v in labels_vals], color=colors, alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels([v[0] for v in labels_vals], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Power Law Exponent')
ax.set_title('E. Exponent: Uncorrected vs Corrected', fontweight='bold')
ax.axhline(-1, color='gray', linestyle=':', alpha=0.4, label='1/d reference')
ax.grid(True, alpha=0.2, axis='y')
for i, (label, slope, r) in enumerate(labels_vals):
    ax.text(i, slope - 0.05, f'{slope:.2f}', ha='center', fontsize=9, fontweight='bold')

# F: Cumulative corrected benefit
ax = axes[1, 2]
h_cum = np.cumsum(h_corrected_marg)
a_cum = np.cumsum(a_corrected_marg)
# Normalize
if h_cum[-1] > 0:
    h_cum_norm = h_cum / h_cum[-1]
else:
    h_cum_norm = h_cum
if a_cum[-1] > 0:
    a_cum_norm = a_cum / a_cum[-1]
else:
    a_cum_norm = a_cum
ax.plot(common_x[1:], h_cum_norm, 'b-', linewidth=2, label='Human')
ax.plot(common_x[1:], a_cum_norm, 'r--', linewidth=2, label='AI')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Fraction of Total Corrected Benefit')
ax.set_title('F. Corrected Cumulative Curve', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# Find 50% points
h_half = np.argmin(np.abs(h_cum_norm - 0.5)) + 1
a_half = np.argmin(np.abs(a_cum_norm - 0.5)) + 1
print(f"\nCorrected 50% benefit: Human at {h_half} tokens, AI at {a_half} tokens")

plt.suptitle('Fine-Grained Influence v3: Corrected Power Law (Intact - Shuffled)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig3_finegrain_corrected.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\n" + "="*60)
print("SUMMARY: Corrected vs Uncorrected Exponents")
print("="*60)
for label, slope, r in labels_vals:
    print(f"  {label:<25}: alpha = {slope:.3f} (r = {r:.3f})")